<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Bellek Açısından Verimli Model Ağırlığı Yükleme

- Bu not defteri, GPU (veya CPU) belleği sınırlı olduğunda daha büyük önceden eğitilmiş ya da ince ayarlanmış modelleri yüklemek için ipuçları sunar
- Özellikle, modeli `torch.save(model.state_dict(), "model.pth")` ile kaydettiğiniz (örneğin 5-7. bölümlerde) ve daha sonra ön eğitime devam etmek ya da ek ince ayar yapmak için yeni bir oturumda yüklemek istediğiniz durumlara odaklanır
- Örnek bir LLM kullansa da, bu not defterinde açıklanan yöntemler geneldir ve yalnızca LLM'lere değil, herhangi bir PyTorch modelini yüklemeye uygulanır

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/memory-efficient-loading/memory-efficient-loading.webp" width="800px">

In [1]:
from importlib.metadata import version

pkgs = [
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

torch version: 2.9.1+cu130


&nbsp;
## 1. Ölçüm yardımcıları

- Önce VRAM'i (GPU belleğini) izlemek için bazı yardımcı kodlar tanımlayalım
- Daha sonra ana sistem RAM'ini (CPU belleğini) izleyen bir araç da tanıtacağız
- Bu fonksiyonların amacı, ilerleyen kısımlarda onları uyguladığımızda netleşecek

In [2]:
import gc
import time
import torch


def start_memory_tracking():
    """Initialize GPU memory tracking."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    else:
        print("This notebook is intended for CUDA GPUs but CUDA is not available.")

def print_memory_usage():
    max_gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)  # Convert bytes to GB
    print(f"Maximum GPU memory allocated: {max_gpu_memory:.1f} GB")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)  # some buffer time to allow memory to clear
    torch.cuda.reset_peak_memory_stats()
    max_memory_allocated = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    print(f"Maximum GPU memory allocated: {max_memory_allocated:.1f} GB")

&nbsp;
## 2. Model kurulumu

- Bu kod bölümü modelin kendisini kurar
- İşleri daha ilginç kılmak için burada "large" GPT-2 modelini kullanıyoruz (bu not defterinin bellek gereksinimlerini ve çalışma süresini düşürmek için "gpt2-small (124M)" kullanabilirsiniz)

In [3]:
from previous_chapters import GPTModel
# `previous_chapters.py` dosyası yerelde mevcut değilse,
# onu `llms-from-scratch` PyPI paketinden içe aktarabilirsiniz.
# Ayrıntılar için bkz.: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# Ör.:
# from llms_from_scratch.ch04 import GPTModel



BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-xl (1558M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

- Şimdi GPU bellek fonksiyonlarını iş başında görelim:

In [4]:
start_memory_tracking()


model = GPTModel(BASE_CONFIG)
device = torch.device("cuda")
model.to(device)

print_memory_usage()

/home/rasbt/jupyterlab/reasoning/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


Maximum GPU memory allocated: 6.4 GB


- Ayrıca, örnek bir tensör vererek modelin düzgün çalıştığından emin olalım

In [5]:
# Modelin çalışıp çalışmadığını test et (burada belleği izlemeye gerek yok)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

- Ardından, modeli ön eğittiğimizi ve daha sonra kullanmak üzere kaydettiğimizi düşünelim
- Basitlik adına burada gerçek ön eğitimi atlıyor ve yalnızca başlatılmış modeli kaydediyoruz (ancak aynı kavram geçerlidir)

In [6]:
# Eğitim kodu buraya gelecekti...

model.train()
torch.save(model.state_dict(), "model.pth")

- Son olarak, GPU belleğini sıfırlamak için Python oturumundaki modeli ve örnek tensörü siliyoruz

In [7]:
del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 3. Temel ağırlık yükleme

- Şimdi, önceden eğitilmiş model ağırlıklarını yüklediğimiz ilginç kısım başlıyor
- Daha önce kaydedilen modeli yüklemek için ne kadar GPU belleği gerektiğine bakalım

In [8]:
# Sonra önceden eğitilmiş ağırlıkları yükle

start_memory_tracking()

model = GPTModel(BASE_CONFIG)
model.to(device)

model.load_state_dict(
    torch.load("model.pth", map_location=device, weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 12.8 GB


- Belleğin önceki oturumdakinin 2 katı olduğuna dikkat edin
- Bunun nedeni, kısa bir süreliğine aynı modelin bellekte iki kez bulunmasıdır:
  - Birincisi `model.to(device)` aracılığıyla
  - İkincisi `model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))` kod satırı aracılığıyla; sonunda yüklenen model ağırlıkları modele kopyalanacak ve `state_dict` atılacaktır, ancak kısa bir süre boyunca hem ana model hem de yüklenen `state_dict` bellekte bulunur
- Kalan bölümler bu duruma çözüm bulmaya odaklanır
- Ama önce modeli test edip GPU belleğini sıfırlayalım



In [9]:
# Modelin çalışıp çalışmadığını test et (burada belleği izlemeye gerek yok)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- Pratikte çok yaygın olan bir başka kalıbı test edelim:

In [10]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG)

model.load_state_dict(
    torch.load("model.pth", map_location="cpu", weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB


In [11]:
# Modelin çalışıp çalışmadığını test et (burada belleği izlemeye gerek yok)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- Yani tepe bellek açısından bakıldığında, modeli önce cihazda örnekleyip ardından `map_location="device"` kullanmakla, ağırlıkları önce CPU belleğine yükleyip (`map_location="cpu"`) sonra cihaza taşımak arasında bir fark yoktur

&nbsp;
## 4. Ağırlıkları sırayla yüklemek

- Önceki bölümde vurgulanan, model ağırlıklarının GPU belleğinde iki kez bulunması sorununa bir çözüm, modeli sırayla yüklemektir
- Aşağıda şunları yapıyoruz:
  - önce modeli GPU belleğine yüklüyoruz
  - sonra model ağırlıklarını CPU belleğine yüklüyoruz
  - ve son olarak her parametreyi tek tek GPU belleğine kopyalıyoruz



In [ ]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG).to(device)

state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

print_memory_usage()

# Ağırlıkları sırayla modelin parametrelerine kopyala
with torch.no_grad():
    for name, param in model.named_parameters():
        if name in state_dict:
            param.copy_(state_dict[name].to(device))
        else:
            print(f"Warning: {name} not found in state_dict.")

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB


- Yukarıda görebileceğimiz gibi bellek kullanımı öncekinden çok daha düşük
- Belleğin 6,4'ten 6,7 GB'a çıktığına dikkat edin; çünkü başlangıçta bellekte yalnızca model varken, sonrasında model artı 1 parametre tensörü bulunur (parametre tensörünü modele `".to"` ile atayabilmek için geçici olarak GPU'ya taşıyoruz)
- Genel olarak bu kayda değer bir iyileşmedir
- Yine, modeli kısaca test edip bir sonraki bölüm için GPU belleğini sıfırlayalım

In [ ]:
# Modelin çalışıp çalışmadığını test et (burada belleği izlemeye gerek yok)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input, state_dict, param
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 5. Modeli düşük CPU belleğiyle yüklemek

- Önceki oturumda, ağırlıkları (`state_dict`) tek tek modele kopyalamadan önce CPU belleğine yükleyerek GPU bellek kullanımını azalttık
- Peki CPU belleğimiz sınırlıysa ne yaparız?
- Bu bölüm, büyük GPU belleğine ama küçük CPU belleğine sahip makinelerde model yüklemek için PyTorch'un `"meta"` cihaz yaklaşımını kullanır
- Ama önce CPU belleğini izlemek için pratik bir fonksiyon tanımlayalım

In [ ]:
import os
import psutil
from threading import Thread


def memory_usage_in_gb(func, *args, **kwargs):
    process = psutil.Process(os.getpid())

    # Fonksiyonu çalıştırmadan önce temel bellek kullanımını ölç
    baseline_mem = process.memory_info().rss / 1024 ** 3  # in GB

    # Belleği ayrı bir iş parçacığında izlemeye başla
    mem_usage = []
    done = False

    def monitor_memory():
        while not done:
            mem_usage.append(process.memory_info().rss / 1024 ** 3)  # Convert to GB
            time.sleep(0.1)

    t = Thread(target=monitor_memory)
    t.start()

    # Fonksiyonu çalıştır
    func(*args, **kwargs)

    # İzlemeyi durdur
    done = True
    t.join()

    peak_mem_usage_gb = max(mem_usage) - baseline_mem
    return peak_mem_usage_gb


- Başlangıç olarak, önceki bölümdeki sıralı ağırlık yükleme yaklaşımının CPU belleğini izleyelim

In [ ]:
def load_sequentially():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG).to(device)

    state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

    print_memory_usage()

    # Ağırlıkları sırayla modelin parametrelerine kopyala
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_sequentially)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB
-> Maximum CPU memory allocated: 6.3 GB


- Şimdi, düşük CPU belleğine ama büyük GPU belleğine sahip bir makinemiz olduğunu varsayalım
- PyTorch'un "meta" cihazını devreye sokarak CPU belleği ile GPU belleği kullanımı arasında bir ödünleşim yapabiliriz
- PyTorch'un meta cihazı, verileri için gerçek bellek ayırmadan tensör oluşturmanıza olanak tanıyan özel bir cihaz türüdür; böylece etkin biçimde "meta" tensörler oluşturur
- Bu, bellek ayırma yükü olmadan tensör şekillerine ve türlerine ihtiyaç duyduğunuz model analizi veya mimari tanımı gibi görevlerde faydalıdır

In [ ]:
def load_sequentially_with_meta():
    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    state_dict = torch.load("model.pth", map_location=device, weights_only=True)

    print_memory_usage()

    # Ağırlıkları sırayla modelin parametrelerine kopyala
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name])
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(load_sequentially_with_meta)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 1.3 GB


- Yukarıda görebileceğimiz gibi, modeli meta cihazda oluşturup ağırlıkları doğrudan GPU belleğine yükleyerek CPU bellek gereksinimlerini etkin biçimde azalttık
- Şöyle sorulabilir: "O hâlde sıralı ağırlık yükleme hâlâ gerekli mi ve bu, özgün yaklaşımla nasıl karşılaştırılır?"
- Karşılaştırma için basit PyTorch ağırlık yükleme yaklaşımını kontrol edelim (bu not defterindeki ilk ağırlık yükleme bölümünden):

In [ ]:
def baseline():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG)
    model.to(device)

    model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
    model.to(device)
    model.eval();

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(baseline)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 4.4 GB


- Yukarıda görebileceğimiz gibi, meta cihaz olmadan yapılan "basit" ağırlık yükleme daha fazla bellek kullanır
- Başka bir deyişle, sınırlı CPU belleğine sahip bir makineniz varsa, tepe CPU bellek kullanımını azaltmak için model ağırlıklarını doğrudan GPU belleğine yüklemek üzere meta cihaz yaklaşımını kullanabilirsiniz

&nbsp;
## 6. `mmap=True` kullanmak (önerilen)

- Orta veya ileri düzey bir `torch.load` kullanıcısı olarak, bu yaklaşımların PyTorch'taki `mmap=True` ayarıyla nasıl karşılaştırıldığını merak edebilirsiniz
- PyTorch'taki `mmap=True` ayarı, belleğe eşlenmiş (memory-mapped) dosya G/Ç'sini etkinleştirir; bu da tensörün verilere doğrudan disk depolamasından erişmesine olanak tanır, böylece RAM sınırlıysa dosyanın tamamını RAM'e yüklemeyerek bellek kullanımını azaltır
- Ayrıca [mikaylagawarecki](https://github.com/rasbt/LLMs-from-scratch/issues/402) tarafından yazılan faydalı yoruma bakın
- İlk bakışta yukarıdaki sıralı yaklaşımlardan daha az verimli görünebilir:

In [ ]:
def best_practices():
  with torch.device("meta"):
      model = GPTModel(BASE_CONFIG)

  model.load_state_dict(
      torch.load("model.pth", map_location=device, weights_only=True, mmap=True),
      assign=True
  )

  print_memory_usage()

peak_memory_used = memory_usage_in_gb(best_practices)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 5.9 GB


- CPU RAM kullanımının bu kadar yüksek olmasının nedeni, bu makinede yeterince CPU RAM'i bulunmasıdır
- Ancak bunu sınırlı CPU RAM'i olan bir makinede çalıştırsaydınız, `mmap` yaklaşımı daha az bellek kullanırdı

&nbsp;
## 7. Diğer yöntemler

- Bu not defteri, PyTorch'ta ağırlık yüklemenin basit ve yerleşik yöntemlerine odaklanır
- Sınırlı CPU belleği durumları için önerilen yaklaşım, yeterince açıklanan `mmap=True` yaklaşımıdır
- Alternatif olarak, bir diğer seçenek her ağırlık tensörünü ayrı ayrı kaydedip yükleyen kaba kuvvet yaklaşımıdır:

In [ ]:
model = GPTModel(BASE_CONFIG)
# `model` değişkeninin eğitilmiş modeliniz olduğunu varsayalım
state_dict = model.state_dict()

# Tek tek parametre dosyalarını saklamak için bir dizin oluştur
os.makedirs("model_parameters", exist_ok=True)

# Her parametre tensörünü ayrı ayrı kaydet
for name, param in state_dict.items():
    torch.save(param.cpu(), f"model_parameters/{name}.pt")

del model

In [ ]:
def load_individual_weights():

    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    print_memory_usage()
    param_dir = "model_parameters"

    with torch.no_grad():
        for name, param in model.named_parameters():
            weight_path = os.path.join(param_dir, f"{name}.pt")
            if os.path.exists(weight_path):
                param_data = torch.load(weight_path, map_location="cpu", weights_only=True)
                param.copy_(param_data)
                del param_data  # Free memory
            else:
                print(f"Warning: {name} not found in {param_dir}.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_individual_weights)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 0.3 GB
